In [1]:
pip install -q transformers datasets rouge-score sacrebleu torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
pip install -q bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
from sacrebleu import corpus_bleu
from bert_score import score as bert_score   # ✅ ADDED

In [4]:


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================
# 1. LOAD DATASET (SCRIPT-FREE)
# =====================================================
print("Loading dataset...")
dataset = load_dataset(
    "cnn_dailymail",
    "3.0.0",
    split="test[:500]"   # reduce to 100 if too slow
)

texts = dataset["article"]
refs  = dataset["highlights"]

# =====================================================
# 2. MODELS
# =====================================================
models = {
    "PEGASUS": "google/pegasus-cnn_dailymail",
    "BART-CNN": "facebook/bart-large-cnn",
    "FLAN-T5": "google/flan-t5-base"
}

# =====================================================
# 3. METRICS
# =====================================================
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def evaluate_model(name, hf_model):
    print(f"\nEvaluating {name}...")

    tokenizer = AutoTokenizer.from_pretrained(hf_model)
    model = AutoModelForSeq2SeqLM.from_pretrained(hf_model).to(DEVICE)
    model.eval()

    preds = []
    rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    # --------- GENERATION ---------
    for text, ref in tqdm(zip(texts, refs), total=len(texts)):
        if "t5" in hf_model.lower():
            text = "summarize: " + text

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(DEVICE)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_length=128,
                num_beams=4,
                do_sample=False
            )

        pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        preds.append(pred)

        scores = scorer.score(ref, pred)
        for k in rouge_scores:
            rouge_scores[k].append(scores[k].fmeasure)

    # --------- BLEU ---------
    bleu = corpus_bleu(preds, [refs]).score

    # --------- BERTScore (NO TRAINING) ---------
    P, R, F1 = bert_score(
        preds,
        refs,
        lang="en",
        model_type="roberta-large",   # standard choice
        batch_size=16,
        verbose=False
    )

    return {
        "Model": name,
        "ROUGE-1": np.mean(rouge_scores["rouge1"]),
        "ROUGE-2": np.mean(rouge_scores["rouge2"]),
        "ROUGE-L": np.mean(rouge_scores["rougeL"]),
        "BLEU": bleu,
        "BERTScore-F1": F1.mean().item()   # ✅ ADDED
    }

# =====================================================
# 4. RUN EVALUATION
# =====================================================
results = []
for name, model_id in models.items():
    results.append(evaluate_model(name, model_id))

# =====================================================
# 5. RESULTS
# =====================================================
results_df = pd.DataFrame(results).sort_values("ROUGE-L", ascending=False)

print("\n=== MODEL COMPARISON RESULTS ===\n")
print(results_df)

print(f"\n🏆 BEST MODEL (by ROUGE-L): {results_df.iloc[0]['Model']}")

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: unsupported operand type(s) for +: 'Column' and 'list'

In [6]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
from sacrebleu import corpus_bleu
from bert_score import score as bert_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================
# 1. LOAD DATASET
# =====================================================
print("Loading dataset...")
dataset = load_dataset("cnn_dailymail", "3.0.0", split="test[:500]")

texts = list(dataset["article"])          # ← Convert to list
refs  = list(dataset["highlights"])       # ← Critical fix: convert Column to list

# =====================================================
# 2. MODELS
# =====================================================
models = {
    "PEGASUS": "google/pegasus-cnn_dailymail",
    "BART-CNN": "facebook/bart-large-cnn",
    "FLAN-T5": "google/flan-t5-base"
}

# =====================================================
# 3. METRICS
# =====================================================
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def evaluate_model(name, hf_model):
    print(f"\nEvaluating {name}...")

    tokenizer = AutoTokenizer.from_pretrained(hf_model)
    model = AutoModelForSeq2SeqLM.from_pretrained(hf_model).to(DEVICE)
    model.eval()

    preds = []
    rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    # --------- GENERATION ---------
    for text, ref in tqdm(zip(texts, refs), total=len(texts)):
        if "t5" in hf_model.lower():
            text = "summarize: " + text

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=False
        ).to(DEVICE)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_length=128,
                num_beams=4,
                early_stopping=True,
                do_sample=False
            )

        pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        preds.append(pred)

        scores = scorer.score(ref, pred)
        for k in rouge_scores:
            rouge_scores[k].append(scores[k].fmeasure)

    # --------- BLEU ---------
    bleu = corpus_bleu(preds, [refs]).score

    # --------- BERTScore ---------
    P, R, F1 = bert_score(
        preds,
        refs,                     # Now both are list[str]
        lang="en",
        model_type="roberta-large",
        batch_size=16,
        verbose=True
    )

    return {
        "Model": name,
        "ROUGE-1": np.mean(rouge_scores["rouge1"]),
        "ROUGE-2": np.mean(rouge_scores["rouge2"]),
        "ROUGE-L": np.mean(rouge_scores["rougeL"]),
        "BLEU": bleu,
        "BERTScore-F1": F1.mean().item()
    }

# =====================================================
# 4. RUN EVALUATION
# =====================================================
results = []
for name, model_id in models.items():
    results.append(evaluate_model(name, model_id))

# =====================================================
# 5. RESULTS
# =====================================================
results_df = pd.DataFrame(results).round(4).sort_values("ROUGE-L", ascending=False)

print("\n=== MODEL COMPARISON RESULTS ===\n")
print(results_df)

print(f"\n🏆 BEST MODEL (by ROUGE-L): {results_df.iloc[0]['Model']}")

Loading dataset...

Evaluating PEGASUS...


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 500/500 [08:20<00:00,  1.00s/it]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/63 [00:00<?, ?it/s]

computing greedy matching.


...

calculating scores...
computing bert embedding.


  0%|          | 0/63 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/32 [00:00<?, ?it/s]

done in 6.51 seconds, 76.84 sentences/sec

=== MODEL COMPARISON RESULTS ===

      Model  ROUGE-1  ROUGE-2  ROUGE-L     BLEU  BERTScore-F1
0   PEGASUS   0.3548   0.1542   0.2665  12.1232        0.8727
1  BART-CNN   0.3506   0.1476   0.2508  10.5210        0.8720
2   FLAN-T5   0.2189   0.0793   0.1715   4.2999        0.8565

🏆 BEST MODEL (by ROUGE-L): PEGASUS
